# 🚀 Eksperimen Alternatif 4: Interpretasi Fitur Universal Menggunakan SHAP
## Mengungkap Faktor Penentu Flakiness Lintas Proyek vs Dalam Proyek (SHapley Additive exPlanations)

- **Kelompok Peneliti**: Kelompok A01 — Rekayasa Perangkat Lunak A (Magister Teknik Informatika ITS)
- **Hardware Akselerasi**: Laptop GPU NVIDIA GeForce RTX 4050 (6GB GDDR6 VRAM, CUDA 13.4, Driver 617.42)
- **Dataset**: FlakeFlagger (22.236 test cases dari 24 repositori Java, C1 Row-Level Drop: 21.963 baris)
- **Rujukan Paper Utama**:
  1. Lundberg & Lee (NeurIPS 2017 — *A Unified Approach to Interpreting Model Predictions (SHAP)*)
  2. Alshammari et al. (ICSE 2021 — *FlakeFlagger: Predicting Flakiness Without Rerunning Tests*)
  3. Afeltra et al. (IEEE Access 2024 — *A Large-Scale Empirical Investigation Into Cross-Project Flaky Test Prediction*)

### 1. Verifikasi Akselerasi GPU NVIDIA RTX 4050 & TreeExplainer SHAP

In [ ]:
import sys
from pathlib import Path

# Tambahkan path src ke sys.path
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from check_gpu import check_nvidia_smi, check_python_environment, check_xgboost_and_shap_gpu
check_nvidia_smi()
check_python_environment()
check_xgboost_and_shap_gpu()

### 2. Pemuatan & Eksplorasi Dataset FlakeFlagger (C1: Row-Level Drop)

In [ ]:
from data_loader import load_dataset, ALL_PREDICTIVE_FEATURES, TAXONOMY_MAP

df, features, projects = load_dataset("C1")
print(f"Total baris valid: {len(df):,}")
print(f"Total fitur asli FlakeFlagger: {len(features)}")
print(f"Jumlah proyek: {len(projects)}")
print(f"Total flaky tests: {df['flaky'].sum():,} ({df['flaky'].mean()*100:.2f}%)")

### 3. Pemuatan Hasil Ekstraksi SHAP (Cross-Project LOPO vs Within-Project Pooled)

In [ ]:
import pandas as pd
import pickle
import json

results_dir = Path("../results").resolve()
comp_df = pd.read_csv(results_dir / "feature_rank_comparison.csv")

print("=== TOP 10 FITUR PALING UNIVERSAL PADA SKENARIO CROSS-PROJECT ===")
print(comp_df.head(10)[["feature", "dimension", "cross_rank", "within_rank", "rank_shift", "cross_importance_pct"]])

### 4. Analisis Korelasi Peringkat Fitur (Spearman Rank Correlation)

In [ ]:
with open(results_dir / "rank_comparison_stats.json", "r") as f:
    stats_res = json.load(f)

print(f"Spearman's Rank Correlation (rho) : {stats_res['spearman_rho']:.4f}")
print(f"Spearman p-value                   : {stats_res['spearman_pval']:.4e}")
print(f"Pearson Correlation (r)            : {stats_res['pearson_r']:.4f}")
print(f"Rata-rata Rank Shift Test Smells  : +{stats_res['test_smells_shift']['mean_rank_shift']:.2f} peringkat")
print(f"Test Smells yang Naik Peringkat   : {stats_res['test_smells_shift']['smells_promoted']} dari 8 fitur")

### 5. Dekomposisi Kontribusi 3 Dimensi Taksonomi Fitur

In [ ]:
dim_df = pd.DataFrame(stats_res["dimension_contributions"])
print("=== PROPORSI KONTRIBUSI TAKSONOMI FITUR (% TOTAL SHAP) ===")
print(dim_df[["dimension", "within_importance_pct", "cross_importance_pct", "delta_importance_pct"]])

### 6. Menampilkan Visualisasi Publikasi Ilmiah SHAP

In [ ]:
from IPython.display import Image, display
plots_dir = results_dir / "plots"

print("1. SHAP Beeswarm Plot (Cross-Project LOPO-CV):")
display(Image(filename=str(plots_dir / "shap_beeswarm_cross_project_xgb.png")))

print("2. Komparasi Feature Importance (Within vs Cross):")
display(Image(filename=str(plots_dir / "feature_importance_comparison_bar.png")))

print("3. Pergeseran Kontribusi Taksonomi:")
display(Image(filename=str(plots_dir / "taxonomy_dimension_comparison_bar.png")))

print("4. Slopegraph Rank Shift:")
display(Image(filename=str(plots_dir / "rank_shift_slopegraph.png")))

### 7. Studi Kasus Penjelasan Lokal (Local Explanations: Spring-Boot)

In [ ]:
print("Waterfall Plot Kasus Flaky Test:")
display(Image(filename=str(plots_dir / "waterfall_flaky_case_1.png")))

print("Waterfall Plot Kasus Non-Flaky Test:")
display(Image(filename=str(plots_dir / "waterfall_nonflaky_case_1.png")))

### 8. Kurva Ketergantungan SHAP (Dependence Plots & Developer Risk Thresholds)

In [ ]:
print("Dependence Plot ExecutionTime:")
display(Image(filename=str(plots_dir / "dependence_ExecutionTime.png")))

print("Dependence Plot numCoveredLines:")
display(Image(filename=str(plots_dir / "dependence_numCoveredLines.png")))

print("Dependence Plot resource-optimism:")
display(Image(filename=str(plots_dir / "dependence_resource_optimism.png")))